# **5G Deployment Example**

In this example, we will deploy a 5G network using the default 5G parameters of the Post-5G orchestrator, which are:

- Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
- OAI gNB
- A simulated OAI NR UE

First, we install the required dependencies:

In [ ]:
from pathlib import Path
import os
import sys

REPOSITORY = Path("../../../..").resolve()
# Refresh upstream even when its package version number has not changed.
%pip install --upgrade --force-reinstall --no-deps "orchestrator @ git+https://gitlab.inria.fr/slices-ri/blueprints/post-5g/orchestrator.git@main"
# CSI support is in this repository (branch csi): install it in editable mode.
%pip install -e "{REPOSITORY}" --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]


## Install Dependencies:

Install the SLICES CLI used for the BI to create VMs, and for POS to provision baremetal servers:

In [ ]:
# %pip install slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/


Install the SLICES POS CLI:

In [ ]:
# %pip install slices-cli-pos --index-url https://gitlab.inria.fr/api/v4/projects/68732/packages/pypi/simple --extra-index-url https://doc.slices-ri.eu/pypi/


Authenticate with your SLICES account:

In [ ]:
#! slices auth login

Install the Ansible dependencies:

In [ ]:
!{sys.executable} -m ansible.cli.galaxy install -r ../../../../ansible/requirements.yaml

Make sure you have an SSH key that you add to SLICES using the command:

```bash
slices pubkey register /path/to/ssh/key
```

which will be use to grant SSH access to the provisioned SLICES machines.

## Input 5G Network Description:

The Post-5G orchestrator keeps the logical network description separate from the deployment-specific choices and evolving deployment state, so the same experiment can be adapted to different implementations or testbeds.

To achieve that, it uses two separate models:
- The Post-5G metamodel describing the entire network and its relationships using in pure 3GPP-standard way
- The deployment model describing implementation choices, target nodes, provider settings, and operational state

To deploy, you invoke lifecycle managers for the work you need. For example, the `ProvisioningManagement` class reserves or creates resources, `KubernetesManagement` prepares nodes and forms a cluster, `CoreManagement`, `RANManagement`, `RICManagement`, and `UEManagement` prepare and install workloads on their respective 5G components.

These stages are composable management operations, not necessarily one automatic "deploy everything" command.

Each stage exposes its particular set of actions that run the corresponding Ansible roles and Helm actions, often concurrently where possible.

To get a Meta Model, we can initiate an empty `MetaModel` object that will use a default set of parameters to provide a 3GPP-standard 5G network description:

In [ ]:
from metamodel.models.metamodel import MetaModel

metamodel = MetaModel()

To see the full content of this default Meta Model:

In [ ]:
print(metamodel.to_yaml(exclude_defaults=False))

By default, the metamodel contains:

- Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
- OAI gNB
- A simulated OAI NR UE

Next, to get the deployment model, we need initiate a `DeploymentModel` object.

The easiest way to do so is by providing a YAML input that describes our deployment.

This example explicitly provisions four large VMs: node1 for the core/control plane, node2 for the RAN, node3 for UEs, and node4 for monitoring and the traffic server.

However, we will have to modify some parameters to ensure the VM provisioning is configured correctly.

In [ ]:
from orchestrator.models.deployment import DeploymentModel
from metamodel.models.metamodel import MetaModel

metamodel = MetaModel()
deployment_model = DeploymentModel.from_yaml(
"""
nodes:
  node1: {roles: [core, database]}
  node2: {roles: [ran]}
  node3: {roles: [ue]}
  node4: {roles: []}
ran:
  components:
    gnb: {type: gnb, node_name: node2}
ues:
  sim_ue: {type: containerized, implementation: oai, node_name: node3}
params:
  k8s:
    master_node: node1
  provisioning:
    project: post5g-beta
    experiment: 5g
    extra_path: ~/slices/bin
"""
)

Here, we specified:
- The absolute path to where the SLICES CLI was installed on the Python venv
- The SLICES project to use: `beta`
- The SLICES experiment: `5g`

Make sure to create the SLICES experiment beforehand using:

```bash
slices project use beta
slices experiment create 5gbp-exp --duration 1d
```

or add `create_experiment: true` to your deployment model `params.provisioning` to create the experiment automatically.

By default, the created experiment duration is 1 day, and the VM duration is 3 hours. To change that, use the `experiment_duration` and `node_duration` parameters respectively.

> *Note*: Check if your Ansible roles path was found automatically:

In [ ]:
deployment_model.params.ansible.roles_paths

> *Note*: If the list is empty, your Ansible roles path wan not found. If so, before continuing, please also add the absolute path to your Ansible roles in the `params.ansible.roles_paths` list.

## Provisioning:

To provision the deployment VMs, we will invoke the `ProvisioningManagement` class from the Post-5G orchestrator and initiate it with the `metamodel` and `deployment_model`:

In [ ]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

> *Note*: During the initialization phase of each deployment stage management class, a reconciliation phase ensures the deployment model is  synchronized with the meta model, which takes precedence. 

To provision the nodes, we run the `provision()` action provided by this deployment management class, which will internally run the corresponding `provisioning/bi/provision` Ansible role:

In [ ]:
# Let the BI CLI create the experiment once before provisioning in parallel.
first_node = next(iter(prov.deployment_model.nodes))
await prov.provision_nodes(nodes=[first_node])
await prov.provision()

> *Note*: All the deployment actions are asynchronous function that have to be awaited.

In [ ]:
prov.output_merged(path="merged.yaml", output="yaml")

Once the provisioning is finished, the deployment model will be updated to contain the SSH information about the provisioned nodes needed to generate an Ansible inventory used for the subsequent deployment stages.

In [ ]:
print(prov.get_ansible_inventory())

You can run commands on the provisioned nodes using the `run_node_command` function. For example:

In [ ]:
print(
    (await prov.run_node_command(command=["apt", "update"], nodes=["node1"], become=True, quiet=True, flatten=False))[
        "node1"
    ]
)


> *Note*: The `run_ue_command` can abe used to run commands on provisioned COTS UEs that allow SSH access.

## Kubernetes Deployment:

Next, to deploy Kubernetes on the provisioned nodes, we will invoke the `KubernetesManagement` class.

To initiate it, we can either use the `output()` function of the previous deployment stage to get the metamodel (which is unchanged) and the deployment model (which now contains the SSH info about the nodes) output, and then use them as input just like we did before for the `ProvisioningManagement`.

Or more conveniently, we can give the entire previous deployment class as input for the next one like this:

In [ ]:
from orchestrator.deployment import K8SManagement

k8s = K8SManagement(preceding=prov)

To deploy Kubernetes, we will call the `deploy()` action function of the `KubernetesManagement` class, which is a wrapper function that internally calls the `prepare()` actions to install the necessary dependencies on all the provisioned nodes, then the `create()` action to initiate the Kubernetes cluster on the master node (by default the first node in the `nodes` section of the deployment model) and get the Kubernetes join command, and finally the `join()` action to add the remaining worker nodes on the newly created cluster.

Again, these actions execute internally the corresponding `k8s/<action>` ansible role.

In [ ]:
await k8s.deploy()

Once the Kubernetes cluster is created we can move to the actual 5G deployment.

## 5G Core Deployment:

To deploy the 5G core network, we again invoke the corresponding `CoreManagement` class and initiate it using its preceding class: 

In [ ]:
from orchestrator.deployment import CoreManagement

core = CoreManagement(preceding=k8s)

> *Note*: The 5G deployment management classes (`CoreManagement`, `RANMamanegemt`, `RICManagement` and `UEManagement`) all have a reconciliation phase that handles resolving the cross-references between the different 5G components (e.g. gNB referencing the AMF) and populating the container images and other Kubernetes parameters that can also be changed manually through the deployment model.

Again, we call the `deploy()` action function, which internally will prepare the core network components (AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR) and the database by:
- copying the Helm chart file to the master node
- rendering the Jinja templates from the metamodel and deployment model configuration parameters
- installing the Helm chart and getting the installed pod network information
- registering the subscribers in the `ues` section of the metamodel on the 5G core database

In [ ]:
await core.run_database_action(action="prepare")
await core.prepare()


In [ ]:
# Preserve the database and subscriber steps performed by upstream core.deploy().
await core.run_database_action(action="install")
await core.install()
await core.register_ues()


## RAN Deployment:

Once the 5G core has been deployed, the deployment model should now contain the necessary network information about the AMF for the gNB to connect to it.

To deploy the OAI RFSIM RAN, we again call the corresponding `RANManagement` class and initiate it using the preceding class:

In [ ]:
from orchestrator.deployment import RANManagement

ran = RANManagement(preceding=core)

Then call the `deploy()` action function to prepare and install the gNB Helm chart:

In [ ]:
# Select local OAI charts that include the metrics-parser sidecar.
from importlib.resources import files

oai_charts = files("nador") / "assets" / "ran" / "oai"
for name, component in ran.deployment_model.ran.components.items():
    if (
        component.implementation == "oai"
        and component.type in ("gnb", "cu", "cu_cp", "cu_up", "du")
    ):
        component.custom_chart_path = str(oai_charts / f"oai-{component.type}")
        component.custom_rendered_files = [
            str(oai_charts / "values.yaml.j2"),
            str(oai_charts / "config.yaml.j2"),
        ]


### CSI logging

The gNB (pinned on `node2`) can log SRS channel estimates per resource block. `enable_csi` selects the `r2labuser/oai-gnb-csi` image
with the same OAI tag for the radios of `CSI_COMPONENTS` only (with an empty list, no CSI and the images without
logger), forces periodic SRS and CSI-RS in the rendered configuration, and mounts `/data/csi` from the node: each
logger writes `/data/csi/csi_per_rb.csv` on its own VM, where the file survives pod restarts. Run this cell before
preparing the radios. RFsim estimates are flat across RBs (no fading model, see RFsim channel below).

`CSI_ENV` passes options to the logger (`CSI_GRANULARITY`, `CSI_SUBCARRIER_SAMPLING`, `CSI_ANTENNA_SELECTION`,
`CSI_PORT_SELECTION`, `CSI_FLUSH_CORE`) and `CSI_COLLECT` sets how each experiment retrieves the CSI; the full option
tables are in `examples/monitoring/multi-gnb-2du/test.ipynb`.

In [ ]:
from nador.csi import CsiCollector, enable_csi

# CSI mode: radios that log (empty list: no CSI) and logger options (empty: defaults, RB granularity)
CSI_COMPONENTS = ['gnb']
CSI_ENV = {}  # e.g. {"CSI_GRANULARITY": "subcarrier", "CSI_SUBCARRIER_SAMPLING": "2"}
# CSI retrieval by each ExperimentRun: whole file + per-section files, at the lowest priority on the radio VM
CSI_COLLECT = {"fetch_full": True, "window_types": "section", "padding_seconds": 0}

enable_csi(ran, components=CSI_COMPONENTS, env=CSI_ENV)
for name in CSI_COMPONENTS:
    component = ran.deployment_model.ran.components[name]
    print(f"{name} on {component.node_name}: {component.image.repository}:{component.image.image_tag}",
          component.extra_env_vars)

# One collector per CSI component, reused by every ExperimentRun below.
CSI_COLLECTORS = [CsiCollector(ran, name, **CSI_COLLECT) for name in CSI_COMPONENTS]

#### Default CSI settings

With the CSI cells as provided (`CSI_ENV = {}`), the settings below apply. Change them in the cells named in the last column; logger settings take effect when the radios start, so set them before preparing the radios.

| Setting | Value in this notebook | Where it is set |
|---|---|---|
| Granularity (`CSI_GRANULARITY`) | `rb`: one complex value per RB | `CSI_ENV` (logger default) |
| Subcarrier sampling (`CSI_SUBCARRIER_SAMPLING`) | 1 (only used with `subcarrier`) | `CSI_ENV` (logger default) |
| Antennas / ports (`CSI_ANTENNA_SELECTION` / `CSI_PORT_SELECTION`) | all | `CSI_ENV` (logger default) |
| Writer thread CPU (`CSI_FLUSH_CORE`) | 32 (left unpinned with a warning when the CPU does not exist) | `CSI_ENV` (logger default) |
| SRS period | chosen by OAI: smallest allowed period covering TDD period x maximum number of UEs per gNB (measured in the rfsim scenarios: 160 slots, 80 ms) | OAI gNB/DU (`do_SRS: periodic`) |
| CSI file | `/data/csi/csi_per_rb.csv` on the radio node (hostPath) | `enable_csi` (`output_dir`) |
| Whole CSI file copied per run (`fetch_full`) | yes | `CSI_COLLECT` |
| Windows cut (`window_types`, `padding_seconds`) | `section`, 0 s | `CSI_COLLECT` |
| Reading CPUs on the radio node (`cpu_list`) | any (lowest priority) | `CSI_COLLECT` |
| Temporary files on the radio node (`cleanup_remote`) | removed | `CsiCollector` default |
| Clock offset correction | applied only above 0.5 s (offset and drift recorded per run) | `ExperimentRun(csi_clock_threshold_s=0.5)` |
| Live view | on `node1`, port `8501`, 120 s window, 10 s refresh, live copy restarted beyond 1024 MB, visualizer v9.0 (`oai-csi-logging` `csi-v3.1`) | `CSI_LIVE`, `CSI_LIVE_PORTS` |
| RFsim channel (`RFSIM_CHANNEL`) | `2x2+awgn`: 2 gNB/DU receive antennas, single-port SRS (see RFsim channel) | RFsim channel cell |

### RFsim channel

`RFSIM_CHANNEL` in the next cell selects the radio channel of the OAI gNBs/DUs with a simulated RU and of their UEs:

| `RFSIM_CHANNEL` | gNB/DU antennas | UE antennas (transmit / receive) | Channel model |
|---|---|---|---|
| `"2x2+awgn"` (default) | 2 (2 PUSCH ports, 2 MIMO layers) | 2 / 1 | AWGN, noise power -30 dB, no path loss |
| `"2x2+epa"` (still to be debugged) | 2 (2 PUSCH ports, 2 MIMO layers) | 2 / 1 | 3GPP EPA fading (7 paths up to 410 ns, 2x2 correlation) |
| `"awgn"` | 1 | 1 / 1 | the same AWGN model |
| `"ideal"` | 1 | 1 / 1 | none |

`"2x2+awgn"` is the radio configuration of the 5g_ansible rfsim scenario (`oai5g-rru` `gnb-config-rfsim.yaml` /
`du-config-rfsim.yaml`, UEs with `--ue-nb-ant-tx 2`). The OAI UE takes its antennas from the RU of its configuration,
so they are set on the simulated RU of each radio. Set it before preparing the radios and before the UE manager is
created.

What to expect in the CSI: OAI's AWGN channel is the identity matrix plus white noise (`random_channel.c`), and the
UEs send a single-port SRS (OAI UE capabilities without `--uecap_file`). In `"2x2+awgn"`, gNB/DU antenna 0 receives the
SRS and antenna 1 stays near 0 (it is linked to the second UE antenna, which sends no SRS). The estimates are flat
across RBs in every profile: OAI adds `10**(noise_power_dB/10) * 256` of noise per int16 sample, negligible at -30 dB
(0 dB already prevents UE attachment). `"2x2+epa"` uses a fading model instead: both gNB/DU antennas receive the SRS through different,
frequency-selective channels. RFsim draws the channel once per UE connection (OAI `simulator.cpp`), so the estimates
do not evolve over time for a connection, and the multipath convolution costs much more CPU than AWGN: check that the
radio VMs keep up (frame advance in the CSI, UE attachment).

Known issues with the OAI images used here (`develop-f80a883`, 2026-10-16), observed on SLICES BI:

- `"2x2+epa"` is still to be debugged. The EPA models load (`Model rfsimu_channel_ue0 type EPA allocated`), but the
  gNB/DU softmodem crashes as soon as a UE connects, right after `Random channel rfsimu_channel_ue0 in rfsimulator
  activated`: the kernel log of the radio VM shows `ru_thread: segfault at 0 ... error 4 in librfsimulator.so`, always
  at offset `0xf11b` (NULL pointer read in the rfsimulator when it applies the EPA channel). The UEs then lose their
  connection (`write() failed, errno(104)`) and no UE attaches. The configuration rendered by nador is the same
  as `"2x2+awgn"` except the channel type.
- `"2x2+awgn"`: the UEs attach (gNB/DU 2x2, UEs with 2 transmit and 1 receive antennas), but a UE softmodem may crash
  under sustained traffic (iperf): `DL__actor: segfault ... error 6 in libc.so.6`, always at offset `0x17173a`
  (invalid write while processing the downlink). Use `"awgn"` or `"ideal"` for long traffic experiments until it is
  understood.

In [ ]:
from nador.csi import set_rfsim_channel

# RFsim channel: "2x2+awgn" (default, as the 5g_ansible rfsim scenario), "2x2+epa" (fading, still to be debugged), "awgn" (1x1) or "ideal"
RFSIM_CHANNEL = "2x2+awgn"
set_rfsim_channel(ran, RFSIM_CHANNEL)
for name, component in ran.deployment_model.ran.components.items():
    advanced = component.advanced_config
    if advanced is not None and advanced.channelmod:
        antennas = (component.extra_env_vars or {}).get("RFSIM_ANTENNAS", "1")
        print(f"{name}: {antennas}x{antennas}, {advanced.channelmod['modellist_rfsimu_1'][0]['type']} channel")

In [ ]:
await ran.prepare()

In [ ]:
await ran.install()

For all the available 5G deployment management classes, we can use the `get_logs()` action function that will fetch logs from the deployed 5G components. A single component returns text; multiple components return a dictionary keyed by component name.

> *Note*: By default, the orchestrator fetches the logs from the deployed 5G component since the last 1 hour, but again, these parameters can be changed for each component in the deployment model.

In [ ]:
print(await ran.get_logs(
    components=["gnb"],
    containers=["gnb"],
    quiet=True,
))

> *Note*: The `quiet=True` option is used to suppress the Ansible output to stdout.

## Monitoring before UE attachment

The core and RAN must be running before monitoring is installed. AMF/SMF capture
must be active **before the first UE attaches**, so the mapper sees registration
and session setup. Do not run the UE deployment cells before this section passes.

Create `monitoring.yaml` beside this notebook using the configuration in
[the integration guide](../../../../MONITORING.md). Enable `latency: true` and
`packet_pairing: true`; verify actual AMF/SMF/UPF/gNB container names and capture
interfaces. The Grafana Helm chart manages the default `admin` / `admin` credentials.
`prepare()` resolves namespaces and selectors from the deployed model.

The experiments open their own Prometheus SSH tunnels using the deployment model.
Set the artifact query profile in `QUERIES_FILE` for the iperf experiment.


In [ ]:
from pathlib import Path
import asyncio
import json

from nador import MonitoringManagement
from nador.models import MonitoringDeployment

QUERIES_FILE = Path("queries.json")  # Your Prometheus export query profile.

monitoring = MonitoringManagement(
    preceding=ran,
    config=MonitoringDeployment.from_yaml_file("monitoring.yaml"),
)


In [ ]:
try:
    await monitoring.deploy()
finally:
    monitoring.output(
        metamodel_file="monitoring_metamodel.yaml",
        deployment_model_file="monitoring_deployment_model.yaml",
    )

### Open Grafana from this notebook

Run after monitoring deployment. This forwards localhost port **30096** to the
current control VM's Grafana port **32005**, using the model's SSH jump host.
The tunnel stays open while you use Grafana. Rerunning the cell replaces its
previous tunnel. To close it explicitly, run `grafana_tunnel.terminate()`.


In [ ]:
import atexit
import shlex
import subprocess
from IPython.display import Markdown, display

# Close the previous tunnel before opening one with the current host address.
if "grafana_tunnel" in globals() and grafana_tunnel.poll() is None:
    grafana_tunnel.terminate()
    await asyncio.to_thread(grafana_tunnel.wait, 5)

# get SSH info and fallback
cfg = monitoring.deployment_model.monitoring
control = monitoring.deployment_model.params.k8s.master_node
ssh = monitoring.deployment_model.nodes[control].ssh_info

command = ["ssh", "-N", "-T", "-S", "none", "-o", "ControlMaster=no", "-o", "ControlPersist=no",
           "-o", "BatchMode=yes", "-o", "ExitOnForwardFailure=yes", "-o", "ConnectTimeout=10"]

command += shlex.split(ssh.common_args or "")

if ssh.private_key_file:
    command += ["-i", str(Path(ssh.private_key_file).expanduser())]
command += ["-p", str(ssh.port), "-L", f"127.0.0.1:30096:127.0.0.1:{cfg.grafana_node_port}",
            f"{ssh.user}@{ssh.hostname or control}"]

grafana_tunnel = subprocess.Popen(command, stdout=subprocess.DEVNULL)

# Close the tunnel when the kernel exits normally.
atexit.register(grafana_tunnel.terminate)

GRAFANA_URL = "http://127.0.0.1:30096"
display(Markdown(f"[Open Grafana]({GRAFANA_URL}) — username: **admin**, password: **admin**"))


### Live CSI view

One Streamlit view per CSI radio runs on `node1`, never on a radio VM: it follows the radio VM's CSI file over
SSH, where only `sshd` and `tail -F` run at the lowest priority. The view shows a sliding window (120 s by default) in
local time; start it before attaching UEs so that SRS reports appear as they attach. The first start installs the
viewer (about one minute); the tunnel is opened on this machine. Stop it with the cell before the VMs are destroyed.

In [ ]:
from nador.csi import CsiLiveView
from IPython.display import Markdown, display

CSI_LIVE = {"window_s": 120, "refresh_s": 10, "max_mb": 1024}
CSI_LIVE_PORTS = {'gnb': 8501}
lives = {
    name: CsiLiveView(ran, live_node='node1', component=name, port=CSI_LIVE_PORTS[name], **CSI_LIVE)
    for name in CSI_COMPONENTS
}
links = []
for name, view in lives.items():
    url = await view.start()
    links.append(f"[CSI {name}]({url})")
display(Markdown(" · ".join(links)))

## UE Deployment:

To deploy the OAI RFSIM NR UE defined in the meta and deployment models, we again initiate the corresponding `UEManagement` deployment management class from the previous one to maintain the deployment state so far, which includes the necessary gNB pod network information that will be used to configure and deploy the simulated UE correctly:

In [ ]:
from orchestrator.deployment.ue import UEManagement

ue = UEManagement(preceding=ran)


To deploy, we again call the `deploy()` action function:

In [ ]:
await ue.deploy()

Once deployed, we can again use the `get_logs()` function to check the logs of our UE deployment without having to manually access the Kubernetes pod:

In [ ]:
print(await ue.get_logs(components=["sim_ue"], quiet=True))

### Check the first UE attachment

Inspect the UE logs above, then require an IPv4 address on the simulated UE's
tunnel interface. Adjust `UE_INTERFACE` if your UE image uses another name.
A tunnel address alone does not prove connectivity or correct monitoring data.


In [ ]:
UE_INTERFACE = "oaitun_ue1"
attachment = await ue.run_command(
    command=["ip", "-4", "addr", "show", "dev", UE_INTERFACE], components=["sim_ue"], quiet=True, flatten=False
)
if "inet " not in attachment["sim_ue"]:
    raise RuntimeError("The first UE has no IPv4 tunnel address; inspect attachment logs")
print(attachment["sim_ue"])


Check that each CSI logger runs and that its file grows: the logs show the logger start, and each `csi_per_rb.csv`
must grow between the two listings (the logger flushes every 5 seconds).

In [ ]:
from orchestrator.deployment import ProvisioningManagement

for name in CSI_COMPONENTS:
    logs = await ran.get_logs(components=[name], quiet=True)
    print(f"--- {name}")
    print("\n".join(line for line in logs.splitlines() if "[CSI]" in line or "Init: N_RB_DL" in line))

csi_nodes = {name: ran.deployment_model.ran.components[name].node_name for name in CSI_COMPONENTS}
nodes = ProvisioningManagement(preceding=ran)
for _ in range(2 if CSI_COMPONENTS else 0):  # sizes must grow between the two listings
    listing = await nodes.run_node_command(command=["ls", "-l", "/data/csi/csi_per_rb.csv"],
                                           nodes=sorted(set(csi_nodes.values())), become=True, quiet=True, flatten=False)
    for name, node in csi_nodes.items():
        print(f"{name} ({node}): {listing[node].strip()}")
    await asyncio.sleep(10)

To SSH into the SLICES BI VM and check your deployment, you can run:

```bash
slices bi --infra fr-sophia2-bi-vm1 ssh node1
```

> Make sure you use the same SLICES experiment name used in the deployment model.

From there, you can do:

```bash
sudo su -
```

to switch to the privileged user, and use the standard `kubectl` commands to check your deployment.

In this case, the command to access the UE pod will be:

```bash
kubectl exec deployment/oai-sim-ue -n oai-ue -it -- bash
```

> *Note*: Each 5G component defined in the meta and deployment model follows these naming conventions:
> - namespace: `<implementation>-<category>` (e.g. AMF is in the `core` category and is deployed using the `Open5GS` implementation, so it will be in the `open5gs-core` namespace)
> - chart name: `<implementation>-<name>` (e.g. we deployed an `OAI` gNB called `gnb`, so its deployment name will be `oai-gnb` in the namespace `oai-ran`)

A more convenient option is to use the `run_command` from the deployment management class to execute a command directly on the pod. For example:

In [ ]:
print((await ue.run_command(command=["ip", "addr"], quiet=True, flatten=False))["sim_ue"])


## Adding a new component on the fly:

To add a new 5G component to the metamodel, we can use the `add_component()` action function in the 5G deployment management classes to add a new corresponding 5G component. For example, we can add a UE from the `UEManagement` class we have.

For that, we can use the `UE` meta model to specify a `supi` field different from the default value of `001010000000100`:

In [ ]:
from metamodel.models.ue import UE

ue.add_component(name="sim_ue2", component_meta=UE.from_yaml(
"""
config:
    supi: '001010000000101'
"""
))

This action only adds the UE to the metamodel and calls the synchronize internally to update the deployment model.

To actually add the UE to the 5G network, we first have to register it in the database using the `register_ues()` action function in the `CoreManagement` class:

In [ ]:
core = CoreManagement(preceding=ue)

In [ ]:
await core.register_ues(ues=["sim_ue2"], quiet=True)

Next, we deploy the new `sim_ue2` using the `deploy` action in the `UEManagement` class again:

In [ ]:
# Carry the subscriber-registration stage forward as well.
ue = UEManagement(preceding=core)
await ue.deploy(components=["sim_ue2"], quiet=True)


In [ ]:
# Run only after the traffic experiments: await ue.remove(components=["sim_ue2"])

Once finished, we fetch and print the UE's logs to ensure the deployment was successful:

In [ ]:
print(await ue.get_logs(components=["sim_ue2"], quiet=True))

## TCP iperf from UEs to a Server

The traffic server uses node4, separate from the core, RAN and UE VMs.
The setup cell installs iperf3 on the server VM when needed. Its current SSH address is used as the traffic
endpoint, while each UE binds to its own tunnel address and interface.
Each UE runs 20 seconds uplink, then 20 seconds downlink using reverse mode,
sequentially. One-off VM servers use separate ports starting at 5202, leaving
port 5201 available for an existing iperf service.

The setup cell reads the addresses and ensures the server package is installed. The experiment
cell shows all traffic commands and the artifact collector explicitly. Server
startup is outside traffic windows; JSON reports, configuration, timeline and
Prometheus exports share one result directory. Collection finishes before the
tunnel closes, including on traffic failure. No latency-counter assertion is required.


In [ ]:
from datetime import UTC, datetime
from nador.artifacts import ExperimentRun
from orchestrator.utils.ansible import get_ansible_inventory_line, run_ansible_command, run_ansible_playbook

SERVER_NODE = "node4"
CLIENTS = ["sim_ue", "sim_ue2"]
DURATION, BASE_PORT = 20, 5202
BASELINE_SECONDS, COLLECTION_SECONDS = 5, 5
UE_INTERFACE = "oaitun_ue1"

server_node = ue.deployment_model.nodes[SERVER_NODE]
server_ip = server_node.ssh_info.hostname
server_inventory = get_ansible_inventory_line(SERVER_NODE, server_node)
server_runner = str(Path(ue.deployment_model.params.ansible.runner_path) / "iperf-server")


await run_ansible_playbook(
    playbook=[
        {
            "name": "Install iperf3 on the server VM",
            "hosts": SERVER_NODE,
            "gather_facts": False,
            "become": True,
            "tasks": [
                {
                    "name": "Install iperf3",
                    "ansible.builtin.apt": {
                        "name": "iperf3",
                        "state": "present",
                        "update_cache": True,
                        "cache_valid_time": 3600,
                    },
                }
            ],
        }
    ],
    inventory=server_inventory,
    runner_path=server_runner,
    quiet=True,
)
addresses = await ue.run_command(
    command=["ip", "-j", "-4", "addr", "show", "dev", UE_INTERFACE], components=CLIENTS, quiet=True, flatten=False
)

ue_ips = {}
for name in CLIENTS:
    ipv4 = [a["local"] for device in json.loads(addresses[name]) for a in device["addr_info"] if a["family"] == "inet"]
    if len(ipv4) != 1:
        raise RuntimeError(f"{name}: expected one IPv4 address on {UE_INTERFACE}")
    ue_ips[name] = ipv4[0]

if len(set(ue_ips.values())) != len(CLIENTS):
    raise ValueError("Select UEs with different tunnel addresses")

if not QUERIES_FILE.is_file():
    raise FileNotFoundError(QUERIES_FILE)

print(f"iperf server: {SERVER_NODE} ({server_ip}); UE clients: {ue_ips}")


### 1. Define the traffic commands




In [ ]:
async def start_iperf_server(port):
    await run_ansible_command(
        command=["iperf3", "-s", "-D", "-1", "--idle-timeout", "60", "-B", server_ip, "-p", str(port), "-J"],
        hosts=SERVER_NODE,
        inventory=server_inventory,
        runner_path=server_runner,
        quiet=True,
    )


async def iperf_client(client, direction, duration, port):
    reverse = {"uplink": [], "downlink": ["-R"]}[direction]
    output = await ue.run_command(
        command=[
            "iperf3",
            "-c",
            server_ip,
            "-B",
            ue_ips[client],
            "--bind-dev",
            UE_INTERFACE,
            "-p",
            str(port),
            "-t",
            str(duration),
            "--connect-timeout",
            "10000",
            "-J",
            "--get-server-output",
            *reverse,
        ],
        components=[client],
        quiet=True,
        flatten=False,
    )
    return json.loads(output[client])


### 2. Define your scenario

Each row is one measured step. Add, remove or reorder rows, select a UE and direction,
or set a different duration in seconds. Steps execute sequentially.


In [ ]:
SCENARIO = [
    {"name": "ue1_uplink",   "client": "sim_ue",  "direction": "uplink",   "duration": DURATION},
    {"name": "ue1_downlink", "client": "sim_ue",  "direction": "downlink", "duration": DURATION},
    {"name": "ue2_uplink",   "client": "sim_ue2", "direction": "uplink",   "duration": DURATION},
    {"name": "ue2_downlink", "client": "sim_ue2", "direction": "downlink", "duration": DURATION},
]


### 3. Collect artifacts while executing the scenario

`ExperimentRun` owns the result directory, timeline and configured Prometheus export.
`section(name)` marks a measurement window; `save_json` saves a report in that run.
The server starts before each traffic window. The final wait allows additional
scrapes. The collector opens its own Prometheus tunnel during artifact export
and closes it afterward.

For another workload, use the same `with run.section("step_name"):` pattern around
its command and save its result. The collector itself does not depend on iperf.


In [ ]:
timestamp = datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S.%fZ")
output_directory = Path("results") / f"iperf-{timestamp}"

async with ExperimentRun(
    output_directory,
    name="ue-to-vm-iperf",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
    csi=CSI_COLLECTORS,
) as experiment:
    experiment.save_json("scenario.json", SCENARIO)

    with experiment.section("baseline"):
        await asyncio.sleep(BASELINE_SECONDS)

    for port, test in enumerate(SCENARIO, start=BASE_PORT):
        print(f"Running {test['name']}: {test['client']}, {test['direction']}, {test['duration']} seconds")
        await start_iperf_server(port)

        with experiment.section(test["name"]):
            report = await iperf_client(
                client=test["client"],
                direction=test["direction"],
                duration=test["duration"],
                port=port,
            )

        experiment.save_json(f"{test['name']}.json", report)

    with experiment.section("collection"):
        await asyncio.sleep(COLLECTION_SECONDS)

print(f"Artifacts saved to: {output_directory.resolve()}")


Check the CSI collected by the run: per radio, its node clock (the offset to this machine is only applied above
0.5 s), the rows and 5-second batches read, and the per-section files under `csi/<component>/by_window/section/`.

In [ ]:
meta = json.loads((output_directory / "experiment_metadata.json").read_text())
print(meta["status"], meta["collection_errors"])
for name, m in meta.get("csi", {}).items():  # empty without CSI
    print(f"--- {name} on {m['node']}")
    print("  clock:", {k: m.get(k) for k in ("clock_offset_applied", "clock_offset_applied_s", "clock_drift_s", "warnings")})
    print("  start:", {k: round(m["clock_start"][k], 3) for k in ("offset_s", "rtt_s", "uncertainty_s")})
    print("  data: ", {k: m.get("summary", {}).get(k) for k in ("rows", "markers", "dropped_rows", "windows_written", "windows_empty")})
print(sorted(str(p.relative_to(output_directory)) for p in (output_directory / "csi").rglob("*.gz")))

To retrieve the metamodel and deployment model output of your deployment management classes, you can use the `output()` function:

In [ ]:
ue.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

> *Note*: All your Ansible run logs are stored by action, component and execution timestamp in `~/post-5g/ansible-runner` by default, which you can of course change using the `runner_path` parameter in the `params.ansible` section of the deployment model.

### End of the experiment

**Run All** stops at the next cell, so that the end-of-experiment cells below (stop of the live CSI views, UE
disconnection, release of the nodes) only run when you run them yourself, one at a time. The kernel keeps its state:
run the cells below with Shift+Enter when the experiment is over.

In [ ]:
class StopRunAll(Exception):
    """Ends "Run All" here; IPython shows only the message below (no traceback)."""

    def _render_traceback_(self):
        return [str(self)]


raise StopRunAll("Run All stops here. When the experiment is over, run the cells below one at a time.")

Stop the live CSI views before destroying the VMs: this closes the tunnels and revokes their SSH keys on the radio VMs.

In [ ]:
for view in globals().get("lives", {}).values():
    await view.stop()

To destroy the VMs, we can again invoke the `ProvisioningManagement` class and call the `unprovision` action function:

In [ ]:
await prov.unprovision()

> *Note*: most actions have an opposite action to undo what the original action has done, for example, we have the "remove" action to undo the "deploy" action, the "unprovision" action to undo the "provision" action ... etc.

## MiscellaneousManagement:

The `MiscellaneousManagement` class is the Post-5G orchestrator is a flexible extension point.

It runs custom Ansible playbooks against the current deployment, supplying the merged metamodel and deployment model by default, as well as the default Ansible inventory files used by the built-in roles.

It can collect selected playbook outputs and store them in the deployment model's `miscellaneous` section, to allow experiment-specific tasks that don’t belong to the standard provisioning, Kubernetes, core, RAN, RIC, or UE stages.